## Task 1: Extract Data

In [1]:
import pandas as pd
import requests

In [2]:
# API Key from open weather
API_KEY = "0843a33a66365945526e5e30176bee51"

In [3]:
#Cities to pull weather for
cities = ["Mombasa","Nairobi","London"]

In [4]:
#Base url for openweathers current  weather end point
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

In [5]:
def get_weather(city, api_key):
    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"  # so we get Celsius instead of Kelvin
    }
    response = requests.get(BASE_URL, params=params)
    return response.json()

In [6]:
# Test it on one city first
test_data = get_weather("Mombasa", API_KEY)
test_data

{'coord': {'lon': 39.6636, 'lat': -4.0547},
 'weather': [{'id': 801,
   'main': 'Clouds',
   'description': 'few clouds',
   'icon': '02d'}],
 'base': 'stations',
 'main': {'temp': 27.73,
  'feels_like': 29.06,
  'temp_min': 27.73,
  'temp_max': 27.73,
  'pressure': 1017,
  'humidity': 60,
  'sea_level': 1017,
  'grnd_level': 1014},
 'visibility': 10000,
 'wind': {'speed': 7.33, 'deg': 165, 'gust': 8.75},
 'clouds': {'all': 12},
 'dt': 1784024305,
 'sys': {'country': 'KE', 'sunrise': 1783999791, 'sunset': 1784042645},
 'timezone': 10800,
 'id': 186301,
 'name': 'Mombasa',
 'cod': 200}

In [7]:
from datetime import datetime

def extract_weather_fields(data):
    return {
        "city": data["name"],
        "temperature_c": data["main"]["temp"],
        "humidity_pct": data["main"]["humidity"],
        "weather_condition": data["weather"][0]["main"],
        "weather_description": data["weather"][0]["description"],
        "wind_speed_ms": data["wind"]["speed"],
        "datetime": datetime.utcfromtimestamp(data["dt"])
    }

# Loop through all cities and collect results
weather_records = []

for city in cities:
    raw_data = get_weather(city, API_KEY)
    record = extract_weather_fields(raw_data)
    weather_records.append(record)

# Turn into a DataFrame
weather_df = pd.DataFrame(weather_records)
weather_df

C:\Users\Yvvone\AppData\Local\Temp\ipykernel_5296\3527076522.py:11: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "datetime": datetime.utcfromtimestamp(data["dt"])


,city,temperature_c,humidity_pct,weather_condition,weather_description,wind_speed_ms,datetime
0,Mombasa,27.73,60,Clouds,few clouds,7.33,2026-07-14 10:18:25
1,Nairobi,23.62,44,Clouds,few clouds,3.58,2026-07-14 10:15:18
2,London,22.65,67,Clouds,scattered clouds,2.24,2026-07-14 10:14:28


## Task 2: Transform Data

In [8]:
weather_df.dtypes

city                              str
temperature_c                 float64
humidity_pct                    int64
weather_condition                 str
weather_description               str
wind_speed_ms                 float64
datetime               datetime64[us]
dtype: object

In [9]:
from datetime import timedelta
weather_df["datetime_utc"] = weather_df["datetime"]
weather_df["datetime_eat"] = weather_df["datetime_utc"] + timedelta(hours = 3)

weather_df["date"] = weather_df["datetime_utc"].dt.date
weather_df["time_utc"] = weather_df["datetime_utc"].dt.time
weather_df["time_eat"] = weather_df["datetime_eat"].dt.time


In [10]:
# Drop the original combined datetime column to avoid redundancy
weather_df = weather_df.drop(columns=["datetime"])

weather_df

,city,temperature_c,humidity_pct,weather_condition,weather_description,wind_speed_ms,datetime_utc,datetime_eat,date,time_utc,time_eat
0,Mombasa,27.73,60,Clouds,few clouds,7.33,2026-07-14 10:18:25,2026-07-14 13:18:25,2026-07-14,10:18:25,13:18:25
1,Nairobi,23.62,44,Clouds,few clouds,3.58,2026-07-14 10:15:18,2026-07-14 13:15:18,2026-07-14,10:15:18,13:15:18
2,London,22.65,67,Clouds,scattered clouds,2.24,2026-07-14 10:14:28,2026-07-14 13:14:28,2026-07-14,10:14:28,13:14:28


## Task 3: Loading Data

In [11]:
weather_df.to_csv(r"data/weather_data.csv", index=False)

In [12]:
pd.read_csv(r"data/weather_data.csv")

,city,temperature_c,humidity_pct,weather_condition,weather_description,wind_speed_ms,datetime_utc,datetime_eat,date,time_utc,time_eat
0,Mombasa,27.73,60,Clouds,few clouds,7.33,2026-07-14 10:18:25,2026-07-14 13:18:25,2026-07-14,10:18:25,13:18:25
1,Nairobi,23.62,44,Clouds,few clouds,3.58,2026-07-14 10:15:18,2026-07-14 13:15:18,2026-07-14,10:15:18,13:15:18
2,London,22.65,67,Clouds,scattered clouds,2.24,2026-07-14 10:14:28,2026-07-14 13:14:28,2026-07-14,10:14:28,13:14:28


## Task 4: Basic Analysis

In [13]:
# Hottest city
print("Hottest city:", weather_df.loc[weather_df["temperature_c"].idxmax(), "city"])

Hottest city: Mombasa


In [14]:
# Coolest city
print("Coolest city:", weather_df.loc[weather_df["temperature_c"].idxmin(), "city"])

Coolest city: London


In [15]:
# Most humid City
print("Most humid city:", weather_df.loc[weather_df["humidity_pct"].idxmax(), "city"])

Most humid city: London


In [ ]:
# Temperature range across cities
print("Temperature range across cities:", weather_df["temperature_c"].max() - weather_df["temperature_c"].min(), "°C")

Temperature range across cities: 5.080000000000002 °C


In [ ]:
#  Simple comfort comparison (temp + humidity together)
weather_df["comfort_score"] = weather_df["temperature_c"] + (weather_df["humidity_pct"] / 10)
least_comfortable = weather_df.loc[weather_df["comfort_score"].idxmax(), "city"]
most_comfortable = weather_df.loc[weather_df["comfort_score"].idxmin(), "city"]

print("Likely least comfortable (hot + humid):", least_comfortable)
print("Likely most comfortable (cooler + balanced humidity):", most_comfortable)
print()

Likely least comfortable (hot + humid): Mombasa
Likely most comfortable (cooler + balanced humidity): Nairobi



In [ ]:
#  Wind speed ranking
windiest_city = weather_df.loc[weather_df["wind_speed_ms"].idxmax(), "city"]
calmest_city = weather_df.loc[weather_df["wind_speed_ms"].idxmin(), "city"]

print("Windiest city:", windiest_city, "-", weather_df["wind_speed_ms"].max(), "m/s")
print("Calmest city:", calmest_city, "-", weather_df["wind_speed_ms"].min(), "m/s")
print()

Windiest city: Mombasa - 7.33 m/s
Calmest city: London - 2.24 m/s

